# X-Cell 数据检查

本 notebook 用于检查和探索 X-Cell 相关数据。

**重要**: X-Atlas/Pisces 训练数据和 X-Cell 模型权重目前仍为 Coming Soon。
本 notebook 在数据可用时展示分析流程，数据不可用时给出提示。

包含:
- 对照与扰动表达矩阵
- 一个扰动基因对应的细胞数量
- pseudobulk 表达差异
- top 20 变化基因
- GenePT、ESM-2、STRING 等先验的形状
- X-Cell 的预期输入和输出张量形状

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置路径
base_dir = Path('..')
data_dir = base_dir / 'data'

print(f'数据目录: {data_dir.resolve()}')
print(f'目录存在: {data_dir.exists()}')

# 检查可用数据
print('\n=== 可用数据 ===')
for subdir in ['evaluation', 'priors', 'xatlas_pisces']:
    d = data_dir / subdir
    if d.exists():
        files = [f for f in d.rglob('*') if f.is_file() and not f.name.startswith('.')]
        print(f'{subdir}: {len(files)} 个文件')
        for f in files[:5]:
            print(f'  {f.relative_to(data_dir)} ({f.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'{subdir}: 目录不存在')

## 1. 检查 X-Atlas/Pisces 核心训练数据状态

In [ ]:
# 检查 X-Atlas/Pisces 数据是否可用
xatlas_dir = data_dir / 'xatlas_pisces'
xatlas_files = list(xatlas_dir.glob('*.h5ad')) + list(xatlas_dir.glob('*.parquet')) if xatlas_dir.exists() else []

if xatlas_files:
    print(f'[OK] X-Atlas/Pisces 数据已下载: {len(xatlas_files)} 个文件')
    for f in xatlas_files:
        print(f'  {f.name} ({f.stat().st_size / (1024**3):.2f} GB)')
else:
    print('[INFO] X-Atlas/Pisces 数据尚未公开 (Coming Soon)')
    print('[INFO] 官方链接: https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Pisces')
    print('[INFO] 待官方发布后，运行 download_public_resources.py 即可下载。')
    print()
    print('预期数据规模:')
    print('  - 25.6M 细胞')
    print('  - 7 个 CRISPRi screen')
    print('  - 16 种细胞背景')
    print('  - 细胞系: HCT116, HEK293T, HepG2, iPSC, Jurkat Resting, Jurkat Active, iPSC Multi-Diff')

## 2. 加载外部评测数据（Replogle-Nadig 等）

In [ ]:
try:
    import anndata
    import scanpy as sc
    HAS_SCANPY = True
    print(f'Scanpy 版本: {sc.__version__}')
except ImportError:
    HAS_SCANPY = False
    print('[WARN] scanpy 未安装。请运行: pip install scanpy')

# 查找评测数据
eval_dir = data_dir / 'evaluation'
adata = None

if eval_dir.exists():
    h5ad_files = list(eval_dir.rglob('*.h5ad'))
    if h5ad_files and HAS_SCANPY:
        print(f'\n找到 {len(h5ad_files)} 个 h5ad 文件')
        for f in h5ad_files:
            print(f'  {f.relative_to(eval_dir)} ({f.stat().st_size / (1024**2):.1f} MB)')
        
        # 加载第一个
        print(f'\n加载: {h5ad_files[0]}')
        adata = sc.read_h5ad(h5ad_files[0])
        print(f'形状: {adata.shape} (细胞 × 基因)')
        print(f'obs 列: {list(adata.obs.columns)}')
        print(f'var 列: {list(adata.var.columns)}')
    else:
        print('[INFO] 未找到 h5ad 文件或 scanpy 未安装。')
        print('[INFO] 请先运行 download_public_resources.py 下载评测数据。')
else:
    print('[INFO] 评测数据目录不存在。')

## 3. 对照与扰动表达矩阵分析

In [ ]:
if adata is not None:
    # 识别对照和扰动
    control_keywords = ['control', 'ctrl', 'nt', 'non-targeting', 'wildtype', 'wt']
    
    # 查找扰动列
    perturb_col = None
    for col in adata.obs.columns:
        if any(kw in col.lower() for kw in ['perturb', 'condition', 'guide', 'target']):
            perturb_col = col
            break
    
    if perturb_col:
        print(f'扰动列: {perturb_col}')
        print(f'唯一扰动数: {adata.obs[perturb_col].nunique()}')
        
        # 对照细胞
        control_mask = adata.obs[perturb_col].str.lower().str.contains('|'.join(control_keywords), na=False)
        print(f'对照细胞数: {control_mask.sum()}')
        
        # 每个扰动的细胞数
        perturb_counts = adata.obs[perturb_col].value_counts()
        print(f'\n前10个扰动的细胞数:')
        print(perturb_counts.head(10))
        
        # 选择一个扰动基因分析
        target_perturb = perturb_counts.index[1] if len(perturb_counts) > 1 else perturb_counts.index[0]
        print(f'\n选择分析的扰动: {target_perturb}')
        target_mask = adata.obs[perturb_col] == target_perturb
        print(f'该扰动的细胞数: {target_mask.sum()}')
    else:
        print('未找到明显的扰动列。')
        control_mask = None
        target_mask = None
else:
    print('数据未加载，跳过分析。')

## 4. Pseudobulk 表达差异和 Top 20 变化基因

In [ ]:
if adata is not None and control_mask is not None and target_mask is not None:
    # 计算 pseudobulk
    control_expr = np.array(adata[control_mask].X.mean(axis=0)).flatten()
    target_expr = np.array(adata[target_mask].X.mean(axis=0)).flatten()
    
    # log2 fold change
    pseudocount = 1
    log2fc = np.log2((target_expr + pseudocount) / (control_expr + pseudocount))
    
    # Top 20 变化基因
    top_up_idx = np.argsort(log2fc)[-20:][::-1]
    top_down_idx = np.argsort(log2fc)[:20]
    
    gene_names = adata.var_names
    
    print('=== Top 20 上调基因 ===')
    for i, idx in enumerate(top_up_idx):
        print(f'  {i+1:2d}. {gene_names[idx]:20s} log2FC={log2fc[idx]:.3f}')
    
    print('\n=== Top 20 下调基因 ===')
    for i, idx in enumerate(top_down_idx):
        print(f'  {i+1:2d}. {gene_names[idx]:20s} log2FC={log2fc[idx]:.3f}')
    
    # 火山图
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.scatter(log2fc, -np.log10(control_expr + 1), s=1, alpha=0.3, c='gray')
    ax.scatter(log2fc[top_up_idx], -np.log10(control_expr[top_up_idx] + 1), s=10, c='red', label='Top 20 up')
    ax.scatter(log2fc[top_down_idx], -np.log10(control_expr[top_down_idx] + 1), s=10, c='blue', label='Top 20 down')
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(mean control expression)')
    ax.set_title(f'Pseudobulk differential expression: {target_perturb}')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('数据未加载或未识别到对照/扰动，跳过差异分析。')

## 5. 生物先验嵌入检查（GenePT、ESM-2、STRING）

In [ ]:
priors_dir = data_dir / 'priors'

if priors_dir.exists():
    print('=== 生物先验 ===')
    for prior_name in ['genept', 'esm2', 'string', 'depmap', 'cell_painting', 'scgpt']:
        prior_dir = priors_dir / prior_name
        if prior_dir.exists():
            files = [f for f in prior_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
            if files:
                print(f'\n{prior_name}: {len(files)} 个文件')
                for f in files:
                    size_mb = f.stat().st_size / (1024**2)
                    print(f'  {f.name} ({size_mb:.2f} MB)')
                    
                    # 尝试读取 h5 嵌入
                    if f.suffix in ['.h5', '.hdf5']:
                        try:
                            import h5py
                            with h5py.File(f, 'r') as hf:
                                for key in hf.keys():
                                    if isinstance(hf[key], h5py.Dataset):
                                        print(f'    {key}: shape={hf[key].shape}, dtype={hf[key].dtype}')
                        except Exception as e:
                            print(f'    读取失败: {e}')
            else:
                print(f'{prior_name}: 目录存在但无数据文件')
        else:
            print(f'{prior_name}: 目录不存在')
else:
    print('先验数据目录不存在。')
    print('请运行 download_public_resources.py 下载先验数据。')

## 6. X-Cell 预期输入和输出张量形状

In [ ]:
print('=== X-Cell 模型预期输入/输出 ===')
print()
print('输入张量:')
print('  - 基因表达向量 (X): (batch_size, n_genes)')
print('    - n_genes 通常为 ~20000 (人类蛋白编码基因)')
print('    - 值为归一化后的表达 (CP10K + log1p)')
print()
print('  - 扰动目标 (X_target): (batch_size, n_genes)')
print('    - multi-hot 编码，表示该细胞接受的基因扰动')
print('    - 单扰动: 一个 1；组合扰动: 多个 1')
print()
print('  - 协变量 (X_covar): (batch_size, n_covariates)')
print('    - 细胞类型、批次、条件等')
print()
print('输出张量:')
print('  - 预测表达 (X_pred): (batch_size, n_genes)')
print('    - 模型预测的扰动后基因表达')
print()
print('  - 潜在表示 (Z): (batch_size, latent_dim)')
print('    - X-Cell Mini: latent_dim ~ 512-1024')
print('    - X-Cell Ultra: latent_dim ~ 4096-8192')
print()
print('模型参数:')
print('  - X-Cell Mini: ~55M 参数')
print('  - X-Cell Ultra: ~4.87B 参数')
print()
print('[INFO] 以上为基于论文描述的预期形状。')
print('[INFO] 实际形状以官方代码和模型配置为准。')
print('[INFO] 模型权重目前为 Coming Soon，待发布后可验证实际形状。')

## 7. 资源状态总结

In [ ]:
print('=== X-Cell 数据资源状态总结 ===')
print()
print('核心训练数据:')
print('  - X-Atlas/Pisces: Coming Soon (25.6M 细胞)')
print()
print('模型权重:')
print('  - X-Cell Mini 55M: Coming Soon')
print('  - X-Cell Ultra 4.87B: Coming Soon')
print()
print('外部评测数据:')
print('  - Replogle-Nadig: 可下载')
print('  - Parse-1M: 可下载')
print('  - Tahoe-100M: 可下载')
print('  - melanocyte progenitor: 待确认')
print('  - primary CD4+ T cells: 待确认')
print()
print('生物先验:')
print('  - GenePT: 可下载')
print('  - ESM-2: 可下载')
print('  - STRING: 可下载')
print('  - DepMap 24Q4: 可下载')
print('  - JUMP Cell Painting: 可下载')
print('  - scGPT: 可下载')
print()
print('运行以下脚本检查最新状态:')
print('  python scripts/check_resource_status.py')
print()
print('运行以下脚本下载已公开资源:')
print('  python scripts/download_public_resources.py')